# Расчет электронной теплопроводности κe методом ансамблевого Монте-Карло


## 1. Геометрия и температурный градиент

* Область имеет размеры `Lx × Ly`; перенос считается вдоль `x`.
* Границы-термостаты: `T_left = T + ΔT/2`, `T_right = T - ΔT/2`.
* Макроскопический градиент: `gradT = (T_right - T_left) / Lx`.
* Поперечная координата `y` периодическая.


## 2. Инициализация ансамбля

* Позиции частиц разыгрываются равномерно по `x` и `y`.
* Начальные импульсы разыгрываются при средней температуре `(T_left + T_right) / 2`.
* Для потока тепла используется энергия относительно края зоны: `ε_heat = ε(p) - Δ`. Если в задаче задан химпотенциал, вместо `Δ` нужно использовать `μ`: `ε_heat = ε(p) - μ`.


## 3. Один шаг EMC

Для каждой частицы на шаге `dt`:

1. Вычислить `v = ∂ε/∂p` и текущую энергию `ε(p)`.
2. Накопить статистику потоков:
   * тепловой вклад `q_i = (ε_i - μ) v_x,i`;
   * частичный/зарядовый вклад `j_i = v_x,i` для контроля условия открытой цепи.
3. Сдвинуть координату: `x += v_x dt`, `y += v_y dt`. Движение выполняется даже если на этом шаге произойдет рассеяние.
4. Если частица пересекла `x=0` или `x=Lx`, переинжектировать ее из соответствующего термостата. Новое состояние разыгрывается из распределения входящего потока:

   `P(p) ∝ |v_x(p)| exp[-ε(p)/(kB T_bound)]`,

   причем знак `v_x` должен быть направлен внутрь расчетной области.
5. Обновить импульс внешними полями.
6. Разыграть рассеяние по накопленным вероятностям механизмов и заменить импульс, если событие произошло.


## 4. Расчет κe

* Средний тепловой поток на частицу:

  `<q_particle> = mean[(ε_i - μ) v_x,i]`.

* Для 2D листа поток на единицу ширины:

  `q_2D = n_2D <q_particle>`,

  где `n_2D` — поверхностная концентрация носителей.

* Электронная теплопроводность листа:

  `κ_2D = -q_2D / gradT`.

* Для объемной теплопроводности нужно разделить `κ_2D` на эффективную толщину слоя.

* Обязательная проверка: средний частичный поток `n_2D <v_x>` должен быть близок к нулю. Если он не равен нулю, нужно подобрать компенсирующее поле Зеебека и повторить расчет.


## 5. Что изменено в копии кода

* Код сохранен отдельно в `kappa_el_reworked/probable2D`.
* Граничная инжекция теперь переигрывает обе компоненты импульса из распределения входящего потока.
* Частица движется на каждом шаге, а не только на шагах без рассеяния.
* В статистику добавлены `heat_flux` и `particle_flux`; старый `energy_flux` заменен на тепловой поток относительно `Δ`.
* В файл `output/heat_flux_kappa_avg.txt` пишутся: шаг, тепловой поток 2D, частичный поток 2D, оценка `κ_2D`.
